In [ ]:
"""
Effectiveness of thrombolysis and thrombectomy in early (<6hr) arrivals of
non-mild stroke - discharge mRS 0-2.

CAUSAL FOREST version (EconML CausalForestDML).

This is a re-implementation of the S-learner counterfactual notebook
(`01_thrombolysis_thrombectomy_mRS_2`) as an *orthogonalized* causal forest
(Double ML / R-learner). It keeps the same cohort, features and 5-fold
cross-fitting spirit, but replaces the "flip the treatment feature to a 99999
sentinel and re-predict" S-learner with a doubly-robust causal forest that
provides honest CATE confidence intervals.

Two estimands are produced, because section 1.3 of the original states the key
features of interest are the *times*, while the grouped analysis is built around
treatment *strategy*:

  A. STRATEGY (multi-arm, discrete):
       arms = {neither, thrombolysis_only, thrombectomy_only, both}
       baseline arm = "neither"
       -> CATE of each strategy vs no treatment, on the RISK (probability) scale.

  B. TIMING (continuous dose-response, treated-only):
       For each modality (thrombolysis, thrombectomy) fit a continuous-treatment
       causal forest on the treated subgroup only (NO sentinel), giving the
       average partial effect of *delay* (per minute, and reported per hour) on
       P(good outcome), with heterogeneity by X.

Notes on design choices (see accompanying discussion):
  * The 99999 sentinel is REMOVED. A causal forest cannot use a point-mass +
    continuum mixture as a treatment.
  * Outcome is binary; effects are on the probability (risk-difference) scale
    via discrete_outcome=True with a classifier model_y. This differs from the
    log-odds contrasts in the S-learner notebook (documented, not a bug).
  * stroke_team is high-cardinality; it is passed as `groups` for cluster-robust
    cross-fitting rather than one-hot encoded, and is NOT used as a heterogeneity
    (X) feature. Its confounding role is still captured because it is included in
    the nuisance control set W via a frequency/target-style ordinal encoding.
  * ethnicity is ordinal-encoded for the nuisance models.

Requires: econml>=0.16, lightgbm, shap, scikit-learn, pandas, numpy.
"""

In [ ]:
import os
import warnings

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.preprocessing import OrdinalEncoder

In [2]:
from econml.dml import CausalForestDML

In [7]:
import os
import warnings
warnings.filterwarnings("ignore")
# Silence EconML/joblib progress bars; set VERBOSE_FIT=1 to re-enable.
_VERBOSE = int(os.environ.get("VERBOSE_FIT", "0"))
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

In [8]:
# ---------------------------------------------------------------------------
# 1. Configuration -- identical cohort / features to the S-learner notebook
# ---------------------------------------------------------------------------
DATA_PATH = "../../data/sam3/cleaned_data.csv"
MRS_TARGET = 2  # good outcome = discharge_disability <= 2
SENTINEL = 99999

In [9]:
# Adjustment / control features (same list as the original notebook).
# These act as BOTH heterogeneity features X (clinical, pre-treatment) and,
# together with the encoded categoricals, as nuisance controls W.
HETERO_FEATURES = [
    "prior_disability",
    "stroke_severity",
    "age",
    "congestive_heart_failure",
    "hypertension",
    "diabetes",
    "afib_anticoagulant",
    "any_afib_diagnosis",
]
CATEGORICAL_FEATURES = ["stroke_team", "ethnicity"]

In [10]:
TIME_THROMBOLYSIS = "onset_to_thrombolysis_time"
TIME_THROMBECTOMY = "onset_to_thrombectomy_time"

In [11]:
# ---------------------------------------------------------------------------
# 2. Load + filter -- byte-for-byte the same selection logic as the notebook
# ---------------------------------------------------------------------------
def load_and_filter(path=DATA_PATH):
    data = pd.read_csv(path, low_memory=False)

    for col in CATEGORICAL_FEATURES:
        data[col] = data[col].replace("", "Empty").astype("category")
    numeric_cols = HETERO_FEATURES + [TIME_THROMBOLYSIS, TIME_THROMBECTOMY]
    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    print(f"Initial data shape: {data.shape}")

    data = data.dropna(subset=["discharge_disability"])
    data = data[data["infarction"] == 1]
    data = data[data["onset_to_arrival_time"] < 360]
    data = data[data["perfusion_imaging_used"] == 0]
    data = data[((data["stroke_severity"] > 5) | (data["lvo"] == 1))]

    data[TIME_THROMBOLYSIS] = data[TIME_THROMBOLYSIS].fillna(SENTINEL)
    data[TIME_THROMBECTOMY] = data[TIME_THROMBECTOMY].fillna(SENTINEL)

    both = (data[TIME_THROMBOLYSIS] < SENTINEL) & (data[TIME_THROMBECTOMY] < SENTINEL)
    lysis_after_ectomy = data[TIME_THROMBOLYSIS] > data[TIME_THROMBECTOMY]
    data = data[~(both & lysis_after_ectomy)]

    data = data[(data[TIME_THROMBECTOMY] < 720) | (data[TIME_THROMBECTOMY] == SENTINEL)]
    data = data[(data[TIME_THROMBOLYSIS] < 360) | (data[TIME_THROMBOLYSIS] == SENTINEL)]

    print(f"Final data shape: {data.shape}")
    return data.reset_index(drop=True)

In [12]:
def add_treatment_columns(data):
    """Derive clean binary indicators + a 4-level strategy arm (no sentinel)."""
    got_lysis = (data[TIME_THROMBOLYSIS] != SENTINEL).astype(int)
    got_ectomy = (data[TIME_THROMBECTOMY] != SENTINEL).astype(int)
    data = data.copy()
    data["got_thrombolysis"] = got_lysis
    data["got_thrombectomy"] = got_ectomy

    def strategy(row):
        if row["got_thrombolysis"] and row["got_thrombectomy"]:
            return "both"
        if row["got_thrombolysis"]:
            return "thrombolysis_only"
        if row["got_thrombectomy"]:
            return "thrombectomy_only"
        return "neither"

    data["treatment_group"] = data.apply(strategy, axis=1)
    return data

In [13]:
# ---------------------------------------------------------------------------
# 3. Feature matrices
# ---------------------------------------------------------------------------
def build_matrices(data):
    """
    Returns:
      X  : heterogeneity features (pre-treatment clinical only) for CATE splits
      W  : nuisance controls (X + ordinal-encoded categoricals) for residualising
      y  : binary good-outcome target
      groups : stroke_team codes for cluster-robust cross-fitting
    """
    y = (data["discharge_disability"] <= MRS_TARGET).astype(int).to_numpy()

    # Heterogeneity features: clinical, pre-treatment, interpretable.
    X = data[HETERO_FEATURES].copy()
    # Median-impute any residual NaNs (e.g. stroke_severity had a few missing).
    X = X.fillna(X.median(numeric_only=True))

    # Nuisance controls W = X plus encoded categoricals so treatment/outcome
    # models can adjust for hospital and ethnicity confounding.
    cat_df = data[CATEGORICAL_FEATURES].astype(str).fillna("Empty")
    cat_df = cat_df.replace({"nan": "Empty", "": "Empty"})
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    cat_encoded = enc.fit_transform(cat_df.to_numpy())
    W = np.hstack([X.to_numpy(), cat_encoded])
    assert not np.isnan(W).any(), "W (nuisance controls) still contains NaN"

    # stroke_team as cluster groups for honest, cluster-robust inference.
    groups = data["stroke_team"].astype(str).astype("category").cat.codes.to_numpy()

    return X, W, y, groups

In [14]:
def make_nuisances(discrete_treatment):
    """LightGBM nuisance learners (fast, handle nonlinearity)."""
    model_y = LGBMClassifier(
        n_estimators=400, num_leaves=31, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1,
        verbosity=-1,
    )
    if discrete_treatment:
        model_t = LGBMClassifier(
            n_estimators=400, num_leaves=31, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
            n_jobs=-1, verbosity=-1,
        )
    else:
        model_t = LGBMRegressor(
            n_estimators=400, num_leaves=31, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
            n_jobs=-1, verbosity=-1,
        )
    return model_y, model_t

In [15]:
# ---------------------------------------------------------------------------
# 4A. STRATEGY analysis -- multi-arm discrete causal forest vs "neither"
# ---------------------------------------------------------------------------
def run_strategy_forest(data, X, W, y, groups):
    print("\n" + "=" * 70)
    print("A. STRATEGY (multi-arm discrete): each arm vs 'neither'")
    print("=" * 70)

    # EconML one-hot encodes a discrete treatment internally; the alphabetically
    # first level is the baseline. We force "neither" to sort first.
    arm = data["treatment_group"].map(
        {"neither": "0_neither",
         "thrombolysis_only": "1_thrombolysis_only",
         "thrombectomy_only": "2_thrombectomy_only",
         "both": "3_both"}
    ).to_numpy()

    model_y, model_t = make_nuisances(discrete_treatment=True)
    est = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=True,
        discrete_outcome=True,           # binary mRS good-outcome target
        n_estimators=2000,
        min_samples_leaf=25,
        max_depth=None,
        honest=True,
        cv=5,                            # 5-fold cross-fitting (matches notebook)
        verbose=_VERBOSE,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    est.fit(y, arm, X=X.to_numpy(), W=W, groups=groups)

    baseline = "0_neither"
    other_arms = ["1_thrombolysis_only", "2_thrombectomy_only", "3_both"]

    print("\nDoubly-robust ATE of each strategy vs 'neither' "
          "(risk-difference in P(mRS 0-2)):")
    results = {}
    for a in other_arms:
        ate = est.ate(X.to_numpy(), T0=baseline, T1=a)
        lb, ub = est.ate_interval(X.to_numpy(), T0=baseline, T1=a, alpha=0.05)
        cate = est.effect(X.to_numpy(), T0=baseline, T1=a)
        results[a] = cate
        print(f"  {a[2:]:<20s}: ATE = {float(np.ravel(ate)[0]):+.4f} "
              f"[95% CI {float(np.ravel(lb)[0]):+.4f}, {float(np.ravel(ub)[0]):+.4f}]  "
              f"| CATE mean={cate.mean():+.4f} sd={cate.std():.4f}")

    # SHAP on the CATE model, for one contrast (thrombolysis_only vs neither).
    try:
        shap_values = est.shap_values(X.to_numpy(), feature_names=list(X.columns))
        print("\nSHAP values computed for the strategy CATE model "
              "(keys):", list(shap_values.keys()))
    except Exception as exc:  # SHAP is best-effort
        print(f"\n[warn] SHAP for strategy model skipped: {exc}")

    return est, results

In [16]:
# ---------------------------------------------------------------------------
# 4B. TIMING analysis -- continuous dose-response, treated subgroup only
# ---------------------------------------------------------------------------
def run_timing_forest(data, modality):
    """
    modality: 'thrombolysis' or 'thrombectomy'.
    Fit a continuous-treatment causal forest on the TREATED subgroup only.
    Treatment W = onset-to-treatment time (minutes). NO sentinel rows.
    Estimand: average partial effect (per-minute change in P(good outcome));
    reported per hour of additional delay.
    """
    print("\n" + "=" * 70)
    print(f"B. TIMING (continuous dose-response): {modality} treated-only")
    print("=" * 70)

    time_col = TIME_THROMBOLYSIS if modality == "thrombolysis" else TIME_THROMBECTOMY
    treated = data[data[time_col] != SENTINEL].copy()
    print(f"  Treated subgroup n = {len(treated)}")

    X = treated[HETERO_FEATURES].copy()
    X = X.fillna(X.median(numeric_only=True))

    cat_df = treated[CATEGORICAL_FEATURES].astype(str).fillna("Empty")
    cat_df = cat_df.replace({"nan": "Empty", "": "Empty"})
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    cat_encoded = enc.fit_transform(cat_df.to_numpy())
    W = np.hstack([X.to_numpy(), cat_encoded])
    assert not np.isnan(W).any(), "W (nuisance controls) still contains NaN"

    T = treated[time_col].to_numpy().astype(float)  # minutes, continuous
    y = (treated["discharge_disability"] <= MRS_TARGET).astype(int).to_numpy()
    groups = treated["stroke_team"].astype(str).astype("category").cat.codes.to_numpy()

    model_y, model_t = make_nuisances(discrete_treatment=False)
    est = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=False,
        discrete_outcome=True,
        n_estimators=2000,
        min_samples_leaf=25,
        honest=True,
        cv=5,
        verbose=_VERBOSE,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    est.fit(y, T, X=X.to_numpy(), W=W, groups=groups)

    # Constant marginal (per-minute) effect and per-hour interpretation.
    cme = est.const_marginal_effect(X.to_numpy())          # per minute, per patient
    lb, ub = est.const_marginal_effect_interval(X.to_numpy(), alpha=0.05)
    per_min = float(np.mean(cme))
    print(f"  Average partial effect: {per_min:+.6f} per minute "
          f"= {per_min * 60:+.4f} per HOUR of delay in P(mRS 0-2)")
    print(f"    (per-minute 95% CI mean: [{float(np.mean(lb)):+.6f}, "
          f"{float(np.mean(ub)):+.6f}])")
    print(f"  CATE heterogeneity across patients (per hour): "
          f"sd={cme.std() * 60:.4f}")

    return est

In [17]:
# ---------------------------------------------------------------------------
# 5. Optional cross-check: how well do the nuisance outcome models fit?
# ---------------------------------------------------------------------------
def nuisance_sanity_check(X, W, y, groups):
    """Quick AUC of the outcome nuisance, to compare with the S-learner's ~0.82."""
    from sklearn.model_selection import cross_val_predict
    from sklearn.model_selection import StratifiedGroupKFold

    model_y, _ = make_nuisances(discrete_treatment=True)
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    proba = cross_val_predict(
        model_y, W, y, cv=cv, groups=groups, method="predict_proba", n_jobs=-1
    )[:, 1]
    auc = roc_auc_score(y, proba)
    bal = balanced_accuracy_score(y, (proba >= y.mean()).astype(int))
    print(f"\n[nuisance check] outcome model E[Y|W]: AUC={auc:.4f}, "
          f"balanced acc={bal:.4f} (cf. S-learner XGB AUC ~0.82)")

In [18]:
# ---------------------------------------------------------------------------
# 6. Main
# ---------------------------------------------------------------------------
def main():
    data = load_and_filter()
    data = add_treatment_columns(data)

    print("\nTreatment group counts:")
    print(data["treatment_group"].value_counts())

    X, W, y, groups = build_matrices(data)

    nuisance_sanity_check(X, W, y, groups)

    strat_est, strat_cate = run_strategy_forest(data, X, W, y, groups)
    lysis_est = run_timing_forest(data, "thrombolysis")
    ectomy_est = run_timing_forest(data, "thrombectomy")

    # Persist per-patient CATEs for downstream analysis / plotting.
    out = data[["treatment_group"] + HETERO_FEATURES].copy()
    for a, cate in strat_cate.items():
        out[f"cate_{a[2:]}_vs_neither"] = cate
    out.to_csv("causal_forest_cate_by_patient.csv", index=False)
    print("\nSaved per-patient strategy CATEs -> causal_forest_cate_by_patient.csv")
    print("\nDone.")

In [ ]:
if __name__ == "__main__":
    main()

Initial data shape: (452863, 70)
Final data shape: (76337, 70)

Treatment group counts:
treatment_group
neither              40408
thrombolysis_only    31284
both                  2898
thrombectomy_only     1747
Name: count, dtype: int64


/home/michael/miniconda3/envs/sam3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/michael/miniconda3/envs/sam3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/michael/miniconda3/envs/sam3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/michael/miniconda3/envs/sam3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/michael/miniconda3/envs/sam3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature nam


[nuisance check] outcome model E[Y|W]: AUC=0.8050, balanced acc=0.7362 (cf. S-learner XGB AUC ~0.82)

A. STRATEGY (multi-arm discrete): each arm vs 'neither'

Doubly-robust ATE of each strategy vs 'neither' (risk-difference in P(mRS 0-2)):
  thrombolysis_only   : ATE = +0.0980 [95% CI +0.0181, +0.1779]  | CATE mean=+0.0980 sd=0.0683
  thrombectomy_only   : ATE = +0.0313 [95% CI -2.4598, +2.5224]  | CATE mean=+0.0313 sd=0.8611
  both                : ATE = +0.1117 [95% CI -1.6058, +1.8292]  | CATE mean=+0.1117 sd=0.6038


  0%|                   | 283/229011 [03:34<2882:40]       